In [ ]:
import geopandas as gpd
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.fs as pafs
from shapely import wkt
import os

# ── Output folder ──────────────────────────────────────────────────────────────
out_dir = "../data/processed/ookla"
os.makedirs(out_dir, exist_ok=True)

# ── US bounding box filter ─────────────────────────────────────────────────────
def filter_us(df):
    return df[
        (df['tile_x'] >= -125) & (df['tile_x'] <= -66.5) &
        (df['tile_y'] >=  24.5) & (df['tile_y'] <=  49.5)
    ].copy()

# ── Convert avg_d_kbps to mbps ─────────────────────────────────────────────────
def add_mbps(df):
    df['avg_d_mbps'] = df['avg_d_kbps'] / 1000
    df['avg_u_mbps'] = df['avg_u_kbps'] / 1000
    return df

# ── 1. Process 2023 Q1-Q3 from Parquet (AWS) ──────────────────────────────────
parquet_quarters = [
    ('2023', 'Q1'),
    ('2023', 'Q2'),
    ('2023', 'Q3'),
]

for year, q in parquet_quarters:
    print(f"Processing {year} {q} (Parquet)...")
    
    path = f"../data/raw/ookla/{year}/{q}/mobile_tiles.parquet"
    df   = pd.read_parquet(path)
    
    # filter to US
    df = filter_us(df)
    print(f"  → US tiles: {len(df)}")
    
    # add mbps columns
    df = add_mbps(df)
    
    # convert WKT tile geometry to GeoDataFrame
    df['geometry'] = df['tile'].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df.drop(columns=['tile']), geometry='geometry', crs='EPSG:4326')
    
    # save
    out_path = f"{out_dir}/{year}_{q}.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    print(f"  → Saved to {out_path}")

# ── 2. Process 2023 Q4 → 2025 Q4 from Shapefiles (advisor) ───────────────────
shapefile_quarters = [
    ('2023', 'Q4'),
    ('2024', 'Q1'), ('2024', 'Q2'), ('2024', 'Q3'), ('2024', 'Q4'),
    ('2025', 'Q1'), ('2025', 'Q2'), ('2025', 'Q3'), ('2025', 'Q4'),
]

for year, q in shapefile_quarters:
    print(f"Processing {year} {q} (Shapefile)...")
    
    path = f"../data/raw/ookla/{year}/{q}/gps_mobile_tiles.shp"
    
    if not os.path.exists(path):
        print(f"  ⚠ File not found: {path} — skipping")
        continue
    
    gdf = gpd.read_file(path)
    
    # add mbps columns
    gdf = add_mbps(gdf)
    
    print(f"  → Tiles: {len(gdf)}")
    
    # save
    out_path = f"{out_dir}/{year}_{q}.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    print(f"  → Saved to {out_path}")

print("\nAll quarters processed!")

In [ ]:
import geopandas as gpd
import os

out_dir = "../data/processed/ookla"

for year, q in [('2024', 'Q2'), ('2024', 'Q4'), ('2025', 'Q4')]:
    print(f"Processing {year} {q}...")
    
    path = f"../data/raw/ookla/{year}/{q}/gps_mobile_tiles.shp"
    
    if not os.path.exists(path):
        print(f"  ⚠ File not found: {path}")
        continue
    
    gdf = gpd.read_file(path)
    gdf['avg_d_mbps'] = gdf['avg_d_kbps'] / 1000
    gdf['avg_u_mbps'] = gdf['avg_u_kbps'] / 1000
    
    print(f"  → Tiles: {len(gdf)}")
    
    out_path = f"{out_dir}/{year}_{q}.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    print(f"  → Saved to {out_path}")

print("\nDone!")

In [ ]:
import os

out_dir = "../data/processed/ookla"
files = sorted(os.listdir(out_dir))
print(f"Total quarters processed: {len(files)}")
for f in files:
    size = os.path.getsize(f"{out_dir}/{f}") / (1024*1024)
    print(f"{size:.1f} MB — {f}")

In [ ]:
import geopandas as gpd
import pandas as pd
import os
from shapely import wkt

# ── Output folder ──────────────────────────────────────────────────────────────
out_dir = "../data/processed/ookla"
os.makedirs(out_dir, exist_ok=True)

# ── 1. Process 2023 Q1-Q3 from Parquet (AWS) — needs US filter + geometry ─────
parquet_quarters = [
    ('2023', 'Q1'),
    ('2023', 'Q2'),
    ('2023', 'Q3'),
]

for year, q in parquet_quarters:
    print(f"Processing {year} {q} (Parquet)...")
    
    path = f"../data/raw/ookla/{year}/{q}/mobile_tiles.parquet"
    df   = pd.read_parquet(path)
    
    # filter to US using bounding box
    df = df[
        (df['tile_x'] >= -125) & (df['tile_x'] <= -66.5) &
        (df['tile_y'] >=  24.5) & (df['tile_y'] <=  49.5)
    ].copy()
    print(f"  → US tiles: {len(df)}")
    
    # add mbps columns
    df['avg_d_mbps'] = df['avg_d_kbps'] / 1000
    df['avg_u_mbps'] = df['avg_u_kbps'] / 1000
    
    # convert WKT tile column to geometry
    df['geometry'] = df['tile'].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df.drop(columns=['tile']), geometry='geometry', crs='EPSG:4326')
    
    # save
    out_path = f"{out_dir}/{year}_{q}.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    print(f"  → Saved to {out_path}")

# ── 2. Process 2023 Q4 → 2025 Q4 from Shapefiles (advisor) ───────────────────
#    Already US-filtered — just add mbps columns and save as GeoPackage
shapefile_quarters = [
    ('2023', 'Q4'),
    ('2024', 'Q1'), ('2024', 'Q2'), ('2024', 'Q3'), ('2024', 'Q4'),
    ('2025', 'Q1'), ('2025', 'Q2'), ('2025', 'Q3'), ('2025', 'Q4'),
]

for year, q in shapefile_quarters:
    print(f"Processing {year} {q} (Shapefile)...")
    
    path = f"../data/raw/ookla/{year}/{q}/gps_mobile_tiles.shp"
    
    if not os.path.exists(path):
        print(f"  ⚠ File not found: {path} — skipping")
        continue
    
    gdf = gpd.read_file(path)
    
    # filter to US using geometry centroid
    centroids = gdf.geometry.centroid
    gdf = gdf[
        (centroids.x >= -125) & (centroids.x <= -66.5) &
        (centroids.y >=  24.5) & (centroids.y <=  49.5)
    ].copy()
    
    gdf['avg_d_mbps'] = gdf['avg_d_kbps'] / 1000
    gdf['avg_u_mbps'] = gdf['avg_u_kbps'] / 1000
    
    print(f"  → US tiles: {len(gdf)}")
    
    out_path = f"{out_dir}/{year}_{q}.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    print(f"  → Saved to {out_path}")

print("\nAll quarters processed!")

In [ ]:
import os

out_dir = "../data/processed/ookla"
total = 0
for f in sorted(os.listdir(out_dir)):
    size = os.path.getsize(f"{out_dir}/{f}") / (1024*1024)
    total += size
    print(f"{size:.1f} MB — {f}")
print(f"\nTotal: {total/1024:.1f} GB")

In [ ]:
import geopandas as gpd
import pandas as pd
import os
from shapely import wkt

# ── Output folder ──────────────────────────────────────────────────────────────
out_dir = "../data/processed/ookla"
os.makedirs(out_dir, exist_ok=True)

# ── 1. Process 2023 Q1-Q3 from Parquet (AWS) — needs US filter + geometry ─────
parquet_quarters = [
    ('2023', 'Q1'),
    ('2023', 'Q2'),
    ('2023', 'Q3'),
]

for year, q in parquet_quarters:
    print(f"Processing {year} {q} (Parquet)...")
    
    path = f"../data/raw/ookla/{year}/{q}/mobile_tiles.parquet"
    df   = pd.read_parquet(path)
    
    # filter to US using bounding box
    df = df[
        (df['tile_x'] >= -125) & (df['tile_x'] <= -66.5) &
        (df['tile_y'] >=  24.5) & (df['tile_y'] <=  49.5)
    ].copy()
    print(f"  → US tiles: {len(df)}")
    
    # add mbps columns
    df['avg_d_mbps'] = df['avg_d_kbps'] / 1000
    df['avg_u_mbps'] = df['avg_u_kbps'] / 1000
    
    # convert WKT tile column to geometry
    df['geometry'] = df['tile'].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df.drop(columns=['tile']), geometry='geometry', crs='EPSG:4326')
    
    # save
    out_path = f"{out_dir}/{year}_{q}.gpkg"
    # delete existing file before saving
    if os.path.exists(out_path):
        os.remove(out_path)
    
    # delete existing file before saving
    if os.path.exists(out_path):
        os.remove(out_path)
    gdf.to_file(out_path, driver="GPKG")
    print(f"  → Saved to {out_path}")

# ── 2. Process 2023 Q4 → 2025 Q4 from Shapefiles (advisor) ───────────────────
#    Already US-filtered — just add mbps columns and save as GeoPackage
shapefile_quarters = [
    ('2023', 'Q4'),
    ('2024', 'Q1'), ('2024', 'Q2'), ('2024', 'Q3'), ('2024', 'Q4'),
    ('2025', 'Q1'), ('2025', 'Q2'), ('2025', 'Q3'), ('2025', 'Q4'),
]

for year, q in shapefile_quarters:
    print(f"Processing {year} {q} (Shapefile)...")
    
    path = f"../data/raw/ookla/{year}/{q}/gps_mobile_tiles.shp"
    
    if not os.path.exists(path):
        print(f"  ⚠ File not found: {path} — skipping")
        continue
    
    gdf = gpd.read_file(path)
    
    # filter to US using geometry centroid
    centroids = gdf.geometry.centroid
    gdf = gdf[
        (centroids.x >= -125) & (centroids.x <= -66.5) &
        (centroids.y >=  24.5) & (centroids.y <=  49.5)
    ].copy()
    
    gdf['avg_d_mbps'] = gdf['avg_d_kbps'] / 1000
    gdf['avg_u_mbps'] = gdf['avg_u_kbps'] / 1000
    
    print(f"  → US tiles: {len(gdf)}")
    
    out_path = f"{out_dir}/{year}_{q}.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    print(f"  → Saved to {out_path}")

print("\nAll quarters processed!")

In [ ]:
import os

out_dir = "../data/processed/ookla"
total = 0
for f in sorted(os.listdir(out_dir)):
    size = os.path.getsize(f"{out_dir}/{f}") / (1024*1024)
    total += size
    print(f"{size:.1f} MB — {f}")
print(f"\nTotal: {total/1024:.1f} GB")

In [ ]:
import geopandas as gpd
import pandas as pd

BASE = "../data/processed/ookla"

quarters = [
    ('2023', 'Q1'), ('2023', 'Q2'), ('2023', 'Q3'), ('2023', 'Q4'),
    ('2024', 'Q1'), ('2024', 'Q2'), ('2024', 'Q3'), ('2024', 'Q4'),
    ('2025', 'Q1'), ('2025', 'Q2'), ('2025', 'Q3'), ('2025', 'Q4'),
]

# ── 1. Load all quadkeys ───────────────────────────────────────────────────────
print("Loading quadkeys for all quarters...")
quarter_tiles = {}
for year, q in quarters:
    path = f"{BASE}/{year}_{q}.gpkg"
    gdf = gpd.read_file(path, columns=['quadkey'])
    quarter_tiles[f"{year}_{q}"] = set(gdf['quadkey'].tolist())
    print(f"  {year} {q}: {len(quarter_tiles[f'{year}_{q}'])} tiles")

# ── 2. Compare consecutive quarters ───────────────────────────────────────────
print("\nComparing consecutive quarters...")
results = []

for i in range(1, len(quarters)):
    curr_year, curr_q = quarters[i]
    prev_year, prev_q = quarters[i-1]

    curr_key = f"{curr_year}_{curr_q}"
    prev_key = f"{prev_year}_{prev_q}"

    curr_tiles = quarter_tiles[curr_key]
    prev_tiles = quarter_tiles[prev_key]

    common        = curr_tiles & prev_tiles
    only_in_curr  = curr_tiles - prev_tiles
    only_in_prev  = prev_tiles - curr_tiles

    results.append({
        'quarter'               : f"{curr_year} {curr_q}",
        'prev_quarter'          : f"{prev_year} {prev_q}",
        'total_tiles'           : len(curr_tiles),
        'tiles_in_prev'         : len(common),
        'pct_in_prev'           : round(len(common) / len(curr_tiles) * 100, 1),
        'new_tiles'             : len(only_in_curr),
        'pct_new'               : round(len(only_in_curr) / len(curr_tiles) * 100, 1),
        'dropped_tiles'         : len(only_in_prev),
        'pct_dropped'           : round(len(only_in_prev) / len(prev_tiles) * 100, 1),
    })

    print(f"  {curr_year} {curr_q}: {len(common)/len(curr_tiles)*100:.1f}% tiles existed in previous quarter")

# ── 3. Save results ────────────────────────────────────────────────────────────
df = pd.DataFrame(results)
df.to_csv("../data/results/ookla_tile_persistence.csv", index=False)

print("\n── Summary ───────────────────────────────────────────────")
print(df.to_string(index=False))

In [ ]:
import pandas as pd

out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"

# 5.1.1 - Persistence
print("═══ TILE PERSISTENCE ═══")
persist = pd.read_csv(f"{out_dir}/ookla_tile_persistence.csv")
print(persist.columns.tolist())
print(persist.head(10))

# Ookla summary
print("\n═══ OOKLA USPS SUMMARY ═══")
ookla_sum = pd.read_csv(f"{out_dir}/us_ookla_usps_summary.csv")
print(ookla_sum.columns.tolist())
print(ookla_sum.head())

# FCC USPS summary
print("\n═══ FCC USPS SUMMARY ═══")
fcc_sum = pd.read_csv(f"{out_dir}/fcc_usps_summary.csv")
print(fcc_sum.columns.tolist())
print(fcc_sum.head())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
from matplotlib import cm

out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"

# =============================================================================
# VIRIDIS COLOR PALETTE
# =============================================================================
viridis = cm.get_cmap('viridis')

# Colors for grouped bars
colors = viridis(np.linspace(0.2, 0.8, 3))

retained_color = colors[0]
new_color = colors[1]
dropped_color = colors[2]

# Colors for intersection chart
bar_color = viridis(0.35)
line_color = viridis(0.85)

# ══════════════════════════════════════════════════════════════════════════════
# 5.1.1 PERSISTENCE CHART
# ══════════════════════════════════════════════════════════════════════════════
persist = pd.read_csv(f"{out_dir}/ookla_tile_persistence.csv")

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(persist))
width = 0.25

ax.bar(
    x - width,
    persist['pct_in_prev'],
    width,
    label='Retained from Previous (%)',
    color=retained_color
)

ax.bar(
    x,
    persist['pct_new'],
    width,
    label='New Tiles (%)',
    color=new_color
)

ax.bar(
    x + width,
    persist['pct_dropped'],
    width,
    label='Dropped from Previous (%)',
    color=dropped_color
)

ax.set_xticks(x)
ax.set_xticklabels(
    persist['quarter'],
    rotation=45,
    ha='right',
    fontsize=9
)

ax.set_ylabel('Percentage of Tiles', fontsize=12)
ax.set_title(
    'Ookla Tile Persistence Across Quarters (2023–2025)',
    fontsize=14,
    fontweight='bold'
)

ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(
    f"{out_dir}/rq2_tile_persistence.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

print("Persistence chart saved!")

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"
ookla_dir = r"C:\Users\julak\Documents\broadband_project\data\processed\ookla"

print("\n═══ FCC HEXAGON TO NEAREST OOKLA TILE DISTANCE ANALYSIS ═══")

# Load FCC challenge data
fcc = gpd.read_file(
    r"C:\Users\julak\Documents\broadband_project\data\processed\fcc_us.gpkg"
)

# Fix shifted outcome/technology values if needed
mask = fcc['technology'] == 'Challenge Upheld - Adjudicated by FCC'
fcc.loc[mask, 'outcome'] = 'Challenge Upheld - Adjudicated by FCC'
fcc.loc[mask, 'technology'] = None

# Create one row per challenged H3 hexagon
fcc_hexes = (
    fcc.dissolve(
        by='h3_cell_id',
        aggfunc={
            'h3_cell_state': 'first',
            'challenge_id': 'count'
        }
    )
    .reset_index()
    .rename(columns={'challenge_id': 'n_challenges'})
)

# Project to meters for distance calculation
fcc_proj = fcc_hexes[
    ['h3_cell_id', 'h3_cell_state', 'n_challenges', 'geometry']
].to_crs(epsg=5070)

print(f"Unique challenged hexagons: {len(fcc_proj)}")

quarters = [
    '2023_Q1', '2023_Q2', '2023_Q3', '2023_Q4',
    '2024_Q1', '2024_Q2', '2024_Q3', '2024_Q4',
    '2025_Q1', '2025_Q2', '2025_Q3', '2025_Q4'
]

all_nearest_results = []

for q in quarters:
    print(f"\nProcessing {q}...")

    # Load Ookla data for quarter
    ookla = gpd.read_file(f"{ookla_dir}/{q}.gpkg")
    ookla_proj = ookla.to_crs(epsg=5070)

    # Keep useful Ookla columns
    ookla_cols = ['quadkey', 'avg_d_mbps', 'tests', 'geometry']
    ookla_subset = ookla_proj[ookla_cols].copy()

    # Find nearest Ookla tile for each FCC challenged hexagon
    nearest = gpd.sjoin_nearest(
        fcc_proj[
            ['h3_cell_id', 'h3_cell_state', 'n_challenges', 'geometry']
        ],
        ookla_subset,
        how='left',
        distance_col='nearest_dist_m'
    )

    # Keep only one nearest tile per challenged hexagon
    nearest = (
        nearest.sort_values('nearest_dist_m')
        .drop_duplicates(subset='h3_cell_id', keep='first')
        .copy()
    )

    # Add quarter and distance conversions
    nearest['quarter'] = q.replace('_', ' ')
    nearest['nearest_dist_km'] = nearest['nearest_dist_m'] / 1000
    nearest['nearest_dist_miles'] = nearest['nearest_dist_m'] / 1609.344

    # Flag direct overlap
    nearest['overlaps_ookla_tile'] = nearest['nearest_dist_m'] == 0

    # Keep output columns
    nearest_out = nearest[
        [
            'quarter',
            'h3_cell_id',
            'h3_cell_state',
            'n_challenges',
            'quadkey',
            'avg_d_mbps',
            'tests',
            'nearest_dist_m',
            'nearest_dist_km',
            'nearest_dist_miles',
            'overlaps_ookla_tile'
        ]
    ].copy()

    all_nearest_results.append(nearest_out)

    print(f"  Saved nearest distance rows: {len(nearest_out)}")
    print(f"  Median distance: {nearest_out['nearest_dist_km'].median():.3f} km")
    print(f"  Mean distance: {nearest_out['nearest_dist_km'].mean():.3f} km")
    print(f"  Max distance: {nearest_out['nearest_dist_km'].max():.3f} km")
    print(f"  Overlap percent: {nearest_out['overlaps_ookla_tile'].mean() * 100:.1f}%")

# Combine all quarters
nearest_all = pd.concat(all_nearest_results, ignore_index=True)

# Save all hex-level nearest distances
nearest_all.to_csv(
    f"{out_dir}/rq2_nearest_ookla_distance_by_hex.csv",
    index=False
)

print("\nSaved:")
print(f"  → {out_dir}\\rq2_nearest_ookla_distance_by_hex.csv")

# Optional summary by quarter
summary = nearest_all.groupby('quarter').agg(
    total_challenge_hexes=('h3_cell_id', 'nunique'),
    pct_overlapping_ookla=('overlaps_ookla_tile', lambda x: round(x.mean() * 100, 1)),
    avg_nearest_dist_km=('nearest_dist_km', 'mean'),
    median_nearest_dist_km=('nearest_dist_km', 'median'),
    min_nearest_dist_km=('nearest_dist_km', 'min'),
    max_nearest_dist_km=('nearest_dist_km', 'max'),
    avg_speed_nearest_tile=('avg_d_mbps', 'mean'),
    avg_tests_nearest_tile=('tests', 'mean')
).reset_index()

summary = summary.round({
    'avg_nearest_dist_km': 3,
    'median_nearest_dist_km': 3,
    'min_nearest_dist_km': 3,
    'max_nearest_dist_km': 3,
    'avg_speed_nearest_tile': 1,
    'avg_tests_nearest_tile': 1
})

summary.to_csv(
    f"{out_dir}/rq2_nearest_ookla_distance_summary.csv",
    index=False
)

print(f"  → {out_dir}\\rq2_nearest_ookla_distance_summary.csv")

print("\nSummary:")
print(summary.to_string(index=False))

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

out_dir = r"C:\Users\julak\Documents\broadband_project\data\results"
ookla_dir = r"C:\Users\julak\Documents\broadband_project\data\processed\ookla"

# Choose ONE Ookla quarter/file
ookla_file = f"{ookla_dir}/2025_Q4.gpkg"

print("Loading FCC challenge data...")

fcc = gpd.read_file(
    r"C:\Users\julak\Documents\broadband_project\data\processed\fcc_us.gpkg"
)

# Fix shifted outcome/technology values if needed
mask = fcc['technology'] == 'Challenge Upheld - Adjudicated by FCC'
fcc.loc[mask, 'outcome'] = 'Challenge Upheld - Adjudicated by FCC'
fcc.loc[mask, 'technology'] = None

print(f"FCC challenge rows: {len(fcc)}")

print("Loading Ookla data...")

ookla = gpd.read_file(ookla_file)

# Project both datasets to meters for distance calculation
fcc_proj = fcc.to_crs(epsg=5070)
ookla_proj = ookla.to_crs(epsg=5070)

# Keep useful Ookla columns
ookla_cols = [
    'quadkey',
    'avg_d_mbps',
    'avg_u_mbps',
    'tests',
    'devices',
    'geometry'
]

ookla_subset = ookla_proj[ookla_cols].copy()

print("Finding nearest Ookla tile for each FCC challenge...")

nearest = gpd.sjoin_nearest(
    fcc_proj,
    ookla_subset,
    how='left',
    distance_col='nearest_ookla_dist_m'
)

# Keep one nearest Ookla tile per challenge_id
nearest = (
    nearest.sort_values('nearest_ookla_dist_m')
    .drop_duplicates(subset='challenge_id', keep='first')
    .copy()
)

# Convert distance
nearest['nearest_ookla_dist_km'] = nearest['nearest_ookla_dist_m'] / 1000
nearest['nearest_ookla_dist_miles'] = nearest['nearest_ookla_dist_m'] / 1609.344
nearest['overlaps_ookla_tile'] = nearest['nearest_ookla_dist_m'] == 0

# Rename Ookla columns clearly
nearest = nearest.rename(columns={
    'quadkey': 'nearest_ookla_quadkey',
    'avg_d_mbps': 'nearest_ookla_avg_download_mbps',
    'avg_u_mbps': 'nearest_ookla_avg_upload_mbps',
    'tests': 'nearest_ookla_tests',
    'devices': 'nearest_ookla_devices'
})

# Remove spatial join helper column if present
if 'index_right' in nearest.columns:
    nearest = nearest.drop(columns=['index_right'])

# Save as CSV without geometry
nearest.drop(columns='geometry').to_csv(
    f"{out_dir}/fcc_with_nearest_ookla_tile.csv",
    index=False
)

# Save as GeoPackage with geometry
nearest.to_file(
    f"{out_dir}/fcc_with_nearest_ookla_tile.gpkg",
    driver="GPKG"
)

print("Saved files:")
print(f"  → {out_dir}\\fcc_with_nearest_ookla_tile.csv")
print(f"  → {out_dir}\\fcc_with_nearest_ookla_tile.gpkg")

print("\nPreview:")
print(
    nearest[[
        'challenge_id',
        'h3_cell_id',
        'nearest_ookla_quadkey',
        'nearest_ookla_dist_km',
        'nearest_ookla_avg_download_mbps',
        'nearest_ookla_tests',
        'overlaps_ookla_tile'
    ]].head()
)

In [ ]:
df = pd.read_csv(f"{out_dir}/fcc_with_nearest_ookla_tile.csv")

hex_df = df.drop_duplicates('h3_cell_id')

pct_overlap = hex_df['overlaps_ookla_tile'].mean() * 100
median_miles = hex_df['nearest_ookla_dist_miles'].median()

print(f"{pct_overlap:.1f}% of challenged hexagons overlapped with Ookla tiles")
print(f"median distance to nearest Ookla tile: {median_miles:.2f} miles.")